<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week05-investigation-agents/Nugget027_Investigation_Knowledge_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 026: Persistent Investigation Memory

In [12]:
!pip install -q google-genai
import json

In [13]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [14]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [15]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  resJsonObj  = responseObj[objname]
  return resJsonObj



In [128]:
def save_investigation(memory, investigation):

    memory.append(investigation)

    return memory

In [129]:
def get_last_investigation(memory):

    if len(memory) == 0:
        return None

    return memory[-1]

Load investigation history

Copy data from github project data/investigation_memory.json

In [6]:
data=[
  {
    "investigation_id": 1,
    "date": "2026-06-19",
    "dormant_accounts": 70,
    "inactive_approvers": 35,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH"
  },
  {
    "investigation_id": 2,
    "date": "2026-06-20",
    "dormant_accounts": 75,
    "inactive_approvers": 25,
    "sla_compliance": 70,
    "root_cause": "Workflow Bottleneck",
    "confidence": "HIGH"
  },
  {
    "investigation_id": 3,
    "date": "2026-06-21",
    "dormant_accounts": 65,
    "inactive_approvers": 25,
    "sla_compliance": 70,
    "root_cause": "Workflow Bottleneck",
    "confidence": "HIGH"
  },
  {
    "investigation_id": 4,
    "date": "2026-06-22",
    "dormant_accounts": 85,
    "inactive_approvers": 15,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH"
  }
]

Load data to current notebook context

In [7]:
with open("investigation_memory.json", "w") as f:
    json.dump(data, f)

Load investigation history now

In [8]:
with open("investigation_memory.json") as f:
    investigations = json.load(f)

print(investigations)

[{'investigation_id': 1, 'date': '2026-06-19', 'dormant_accounts': 70, 'inactive_approvers': 35, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH'}, {'investigation_id': 2, 'date': '2026-06-20', 'dormant_accounts': 75, 'inactive_approvers': 25, 'sla_compliance': 70, 'root_cause': 'Workflow Bottleneck', 'confidence': 'HIGH'}, {'investigation_id': 3, 'date': '2026-06-21', 'dormant_accounts': 65, 'inactive_approvers': 25, 'sla_compliance': 70, 'root_cause': 'Workflow Bottleneck', 'confidence': 'HIGH'}, {'investigation_id': 4, 'date': '2026-06-22', 'dormant_accounts': 85, 'inactive_approvers': 15, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH'}]


Build a search function.

In [9]:
def search_by_root_cause(
    investigations,
    root_cause
):

    results = []

    for investigation in investigations:

        if investigation["root_cause"] == root_cause:
            results.append(investigation)

    return results

In [10]:
results = search_by_root_cause(
    investigations,
    "Inactive Approvers"
)

print(results)

[{'investigation_id': 1, 'date': '2026-06-19', 'dormant_accounts': 70, 'inactive_approvers': 35, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH'}, {'investigation_id': 4, 'date': '2026-06-22', 'dormant_accounts': 85, 'inactive_approvers': 15, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH'}]


Create a question-->Retrieve-->Prompt

In [16]:
question = """
Have we seen inactive approver issues before?
"""

In [17]:
historical_cases = search_by_root_cause(
    investigations,
    "Inactive Approvers"
)

In [18]:
prompt = f"""
Question:

{question}

Historical Cases:

{historical_cases}

Answer using historical investigations.
"""

In [20]:
print(callGPT(prompt).text)

Yes, we have seen inactive approver issues before.

According to historical investigations:

*   **Investigation ID 1 (2026-06-19):** The `root_cause` was identified as 'Inactive Approvers'. This investigation involved 35 inactive approvers.
*   **Investigation ID 4 (2026-06-22):** The `root_cause` was also identified as 'Inactive Approvers'. This investigation involved 15 inactive approvers.


Build a generic search

In [21]:
def search_investigations(
    investigations,
    keyword
):

    results = []

    for investigation in investigations:

        if keyword in investigation["root_cause"]:
            results.append(investigation)

    return results

In [22]:
historical_cases = search_investigations(
    investigations,
    "Inactive"
)

In [23]:
prompt = f"""
Question:

{question}

Historical Cases:

{historical_cases}

Answer using historical investigations.
"""

In [25]:
print(callGPT(prompt).text)

Yes, we have seen inactive approver issues before.

Historical cases where 'Inactive Approvers' was identified as the root cause include:

*   **Investigation ID 1 (2026-06-19):** Had 35 inactive approvers and 'Inactive Approvers' was the root cause.
*   **Investigation ID 4 (2026-06-22):** Had 15 inactive approvers and 'Inactive Approvers' was the root cause.
